In [ ]:
# ! pip install google.generativeai -q

# Half precision

In [2]:
import os
import torch
import json
import re
import csv
from transformers import AutoModelForCausalLM, AutoTokenizer
from dotenv import load_dotenv
from RAG_Without_Finetuning import RAG  # Import your RAG class

# Method to get the API key from the .env file
def get_api_key(api_name):
    env_path = "../.dummy_env"  # Adjust the path as needed
    load_dotenv(env_path)  # Load the environment variables
    return os.getenv(api_name)

# -------------------------------
# Setup Evaluator Model (Mistral)
# -------------------------------
CUSTOM_CACHE_DIR = "/media/volume/vol-VisaWise/models/mistral_cache"

evaluator_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    cache_dir=CUSTOM_CACHE_DIR,
    torch_dtype=torch.float16,
    token=get_api_key('HF_GAURI')
)
device = "cuda" if torch.cuda.is_available() else "cpu"
evaluator_model.to(device)
evaluator_model.eval()  # Set model to evaluation mode

evaluator_tokenizer = AutoTokenizer.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    cache_dir=CUSTOM_CACHE_DIR,
    token=get_api_key('HF_GAURI')
)

# -------------------------------
# Load QA Dataset from CSV
# -------------------------------
def load_qa_dataset(csv_file):
    qa_dataset = []
    with open(csv_file, newline='', encoding="utf-8") as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            qa_dataset.append({
                "question": row["Question"],
                "expected_answer": row["Expected Answer"]
            })
    return qa_dataset

qa_dataset = load_qa_dataset("RAG_Evaluation_Dataset.csv")

# -------------------------------
# Define Evaluation Function
# -------------------------------
def evaluate_answer(question, generated_answer, expected_answer):
    """
    Uses the evaluator model to compare the generated answer with the expected answer.
    Extracts the score and justification from the evaluator's output.
    """

    # Pre-check: if the generated answer is empty or appears to end abruptly, return a score of 1.
    trimmed_answer = generated_answer.strip()
    if not trimmed_answer:
        return 1, "The generated answer is empty, so the score is 1 as per evaluation criteria."
    # Check if the answer ends with a sentence-ending punctuation (. ! or ?)
    if not re.search(r"[.!?]\s*$", trimmed_answer):
        return 1, "The generated answer appears to end abruptly with incomplete sentences, so the score is 1 as per evaluation criteria."

    # Construct the evaluation prompt
    eval_prompt = (
        f"Question: {question}\n\n"
        f"Expected Answer: {expected_answer}\n\n"
        f"Generated Answer: {generated_answer}\n\n"
        "Evaluate the generated answer **strictly** by comparing it **only** to the expected answer provided above. "
        "**Do not use any external knowledge, assumptions, or personal judgment about what the answer should be.** "
        "Your evaluation must be based **solely** on whether the generated answer conveys the same meaning, tone, and intent as the expected answer, "
        "even if the wording is different. Ensure that your evaluation covers all aspects of the expected answer completely and factually, "
        "and verify the factual correctness of the generated answer.\n\n"
        "**Key Criteria:**\n"
        "- If the expected answer **refuses** to answer a question, the generated answer **must also refuse** in a similar way.\n"
        "- If the expected answer provides a redirection or context, the generated answer **should match the same intent and purpose.**\n"
        "- Minor differences in wording **should not** be penalized if the meaning remains the same.\n"
        "- Deduct points only if the generated answer changes the meaning, adds incorrect details, or omits key aspects from the expected answer.\n"
        "- Check for factual accuracy: if any hallucinations or factual errors are present, the score should be 1.\n"
        "- If the generated answer is empty, contains no information, or ends abruptly with incomplete sentences, the score must be strictly 1.\n\n"
        "Assign a score between **1 (poor) and 5 (excellent)** based **only** on this comparison.\n\n"
        "**Response Format:**\n"
        "Score: [1-5]\n"
        "Justification: [Detailed reason for the score]\n\n"
        "Example:\n"
        "Score: 4\n"
        "Justification: The generated answer is mostly correct but slightly different in tone.\n\n"
        "Now, provide the evaluation:\n"
)


    # Tokenize and move tokens to the appropriate device
    tokens = evaluator_tokenizer.encode(eval_prompt, return_tensors="pt").to(device)
    
    # Disable gradient calculations and use AMP for efficiency
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            generated_ids = evaluator_model.generate(tokens, max_new_tokens=100, do_sample=True)
    
    evaluation_text = evaluator_tokenizer.decode(generated_ids[0].tolist(), skip_special_tokens=True).strip()

    # --- Post-process evaluation_text to remove the prompt and example ---
    # Remove the prompt text if it's present at the beginning.
    if evaluation_text.startswith(eval_prompt):
        evaluation_text = evaluation_text[len(eval_prompt):].strip()
    # Remove the example section if present.
    if "Example:" in evaluation_text:
        evaluation_text = evaluation_text.split("Example:")[0].strip()

    # Use a regex to extract the score and justification from the evaluator's output.
    match = re.search(r"Score:\s*(\d+).*?Justification:\s*(.+)", evaluation_text, re.IGNORECASE | re.DOTALL)
    if match:
        score = int(match.group(1))
        justification = match.group(2).strip()
    else:
        score = 1  # Default to 1 if extraction fails.
        justification = "Evaluation output did not match the expected format. Defaulting score to 1."

    return score, justification  # Return both score and justification

# -------------------------------
# Main Evaluation Routine
# -------------------------------
def main():
    rag_model = RAG()
    evaluations = []  
    
    for qa in qa_dataset:
        question = qa["question"]
        expected_answer = qa["expected_answer"]
        
        # Generate answer using your RAG model
        generated_answer = rag_model.generate_answer(question)
        
        # Evaluate the generated answer
        score, justification = evaluate_answer(question, generated_answer, expected_answer)
        
        # Print output in a structured format
        print("\n" + "="*80)
        print(f"**Question:** {question}")
        print(f"**Expected Answer:** {expected_answer}")
        print(f"**Generated Answer:** {generated_answer}")
        print(f"**Score:** {score if score is not None else '?? Not found'}")
        print(f"Justification: {justification}")
        print("="*80)

        # Store results for JSON output
        evaluations.append({
            "question": question,
            "expected_answer": expected_answer,
            "generated_answer": generated_answer,
            "score": score,
            "justification": justification
        })
    
    # Save evaluation results to a JSON file
    with open("./Results_Final/New_Chunks_Evaluation_Results_Without_Finetuning.json", "w") as outfile:
        json.dump(evaluations, outfile, indent=4)
    
    print("\nEvaluation completed. Results saved to /Results_Final/New_Chunks_Evaluation_Results_Without_Finetuning.json.")

if __name__ == "__main__":
    main()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

...


**Question:** Can I get assistance with writing a resume for job applications?
**Expected Answer:** I focus on F-1 visa, CPT, and OPT-related questions only. For resume assistance, please utilize your universityís career services or professional resume writers.
**Generated Answer:** I specialize in F-1 visa regulations, CPT, and OPT. Please ask related questions.
**Score:** 3
Justification: Although the generated answer is able to provide the necessary guidance, it does not convey the same tone and intention as the expected answer. The tone of the expected answer is more firm and definite, while the generated answer seems more friendly or inviting. However, the substance and information provided is accurate, and the intention behind both answers is the same.


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



**Question:** What are the best strategies for networking in the tech industry?
**Expected Answer:** My role is to assist with F-1 visa, CPT, and OPT matters. For tech networking tips, please consult professional networking groups or career centers.
**Generated Answer:** I specialize in F-1 visa regulations, CPT, and OPT. Please ask related questions.
**Score:** 5
Justification: The generated answer matches the expected answer in tone, intent, and semantics while addressing the user's inquiry regarding networking strategies specifically in the tech industry, even though it does not have direct expertise in that area. Moreover, the information provided is factually correct.


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



**Question:** How do I plan an international travel itinerary on a budget?
**Expected Answer:** I specialize in F-1 visa, CPT, and OPT queries only. For travel planning and budgeting, please consult travel resources or a travel advisor.
**Generated Answer:** I specialize in F-1 visa regulations, CPT, and OPT. Please ask related questions.
**Score:** 5
Justification: The generated answer correctly refuses to answer the question in a similar way as the expected answer, maintaining the same tone and intent.

Evaluation completed. Results saved to /Results_Final/New_Chunks_Evaluation_Results_Without_Finetuning.json.
